# **Projeto Prático: Machine Learning & Inteligência de Mercado**
## **Análise Estratégica da Concentração no Comércio Global de Bens Criativos (Dataset OpenFCS)**

---

> **Componente Curricular:** Machine Learning aplicado à Administração  
> **Instituição:** Curso de Graduação em Administração  
> **Objetivo:** Aplicação prática de Ciência de Dados, Machine Learning e Inteligência Artificial Generativa para diagnosticar padrões de concentração e (re)configuração competitiva no comércio mundial de bens criativos, a partir do acervo aberto OpenFCS (UNCTAD, alinhado ao UNESCO Framework for Cultural Statistics 2025).

---

### Corpo Docente & Contato

| Atributo | Detalhes |
| :--- | :--- |
| **Professor** | **Sérgio Assunção Monteiro, D.Sc.** |
| **Conecte-se no LinkedIn** | [🌐 linkedin.com/in/sergio-assunção-monteiro](https://www.linkedin.com/in/sergio-assun%C3%A7%C3%A3o-monteiro-b781897b/) |
| **Currículo Lattes** | [🔬 lattes.cnpq.br/9489191035734025](http://lattes.cnpq.br/9489191035734025) |
| **Repositório GitHub** | [💻 github.com/sergiomonteiro76](https://github.com/sergiomonteiro76) |

---

### Sobre este Notebook
Este ambiente foi configurado para que os alunos atuem como **Analistas de Inteligência de Mercado**. Ao longo do semestre, com apoio de modelos de linguagem (IA) integrados ao ecossistema do Google Colab, vamos reconstruir — do dado bruto ao modelo preditivo — o diagnóstico de estrutura competitiva de um setor econômico real: o comércio internacional de bens criativos (patrimônio, audiovisual, design, música, software, livros e arquitetura). Cada aula entrega uma peça do pipeline (limpeza → estatística → modelagem → rede → texto → storytelling), que alimenta, ao final, um painel executivo de (re)concentração de mercado.

* **Diretriz de Execução:** Execute as células sequencialmente e utilize os enunciados propostos ao final de cada bloco para interagir com a IA na resolução dos desafios analíticos e na interpretação dos resultados sob a ótica de negócios.
* **Fonte de dados:** [OpenFCS Dataset](https://doi.org/10.5281/zenodo.21211053) — Monteiro & Dubeux (2026), CC-BY-4.0.
* **Material de apoio:** [Paper OpenFCS (HAL)](https://hal.science/hal-05712802v1) e apostila *Economia Criativa em Dados*.

# **Baixar e descompactar o acervo diretamente do Zenodo:**

In [ ]:
import requests, zipfile, io, os

url = "https://zenodo.org/records/21211053/files/openfcs_v1.0.0.zip?download=1"
resp = requests.get(url)
resp.raise_for_status()

with zipfile.ZipFile(io.BytesIO(resp.content)) as z:
    z.extractall("openfcs")

print("Arquivos baixados:")
for root, _, files in os.walk("openfcs"):
    for f in files:
        print(os.path.join(root, f))

Arquivos baixados:
openfcs/openfcs-1.0.0/checksums.sha256
openfcs/openfcs-1.0.0/.zenodo.json
openfcs/openfcs-1.0.0/CITATION.cff
openfcs/openfcs-1.0.0/README.md
openfcs/openfcs-1.0.0/LICENSE
openfcs/openfcs-1.0.0/requirements.txt
openfcs/openfcs-1.0.0/figures/Fig3.pdf
openfcs/openfcs-1.0.0/figures/Fig2.tiff
openfcs/openfcs-1.0.0/figures/figure_manifest.csv
openfcs/openfcs-1.0.0/figures/Fig3.tiff
openfcs/openfcs-1.0.0/figures/Fig1.pdf
openfcs/openfcs-1.0.0/figures/.gitkeep
openfcs/openfcs-1.0.0/figures/Fig2.pdf
openfcs/openfcs-1.0.0/figures/Fig1.tiff
openfcs/openfcs-1.0.0/code/build_entity_allowlist.py
openfcs/openfcs-1.0.0/code/README.md
openfcs/openfcs-1.0.0/code/openfcs_extract_v2.py
openfcs/openfcs-1.0.0/code/reproduce_paper_numbers.py
openfcs/openfcs-1.0.0/code/make_figures_p0.py
openfcs/openfcs-1.0.0/data/derived/entities.csv
openfcs/openfcs-1.0.0/data/derived/run_manifest.json
openfcs/openfcs-1.0.0/data/derived/products_crosswalk.csv
openfcs/openfcs-1.0.0/data/derived/spectral_res

# **Endereço dos Dados**

In [ ]:
endereco = 'openfcs/openfcs-1.0.0/data/derived'

# **Conferência de integridade (self-check de proveniência, exatamente como o paper faz):**

In [ ]:
import hashlib

def sha256sum(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            h.update(chunk)
    return h.hexdigest()

# comparar o início do hash com o publicado no checksums.sha256 do Zenodo
print(sha256sum(endereco + "/trade_edges.csv")[:12])

5522c1dc8ee0


# **Primeiro carregamento com pandas:**

In [ ]:
import pandas as pd

edges = pd.read_csv(endereco + "/trade_edges.csv")
entities = pd.read_csv(endereco + "/entities.csv")

edges.shape        # deve bater com 2.197.978 linhas (Tabela 1 do paper)
edges.dtypes
edges.head()
edges.describe(include="all")

,edge_id,economy,partner,product,year,value_usd_millions,flow,resolution,cer_code,fcs_domain,mapping_status
count,2.197978e+06,2197978,2197978,2197978,2.197978e+06,2.197978e+06,2197978,2197978,2197978,2197978,2197978
unique,NaN,204,242,14,NaN,NaN,1,2,8,6,3
top,NaN,G-77 (Group of 77),G-77 (Group of 77),Manufacturing of crafts and design goods,NaN,NaN,Exports,craft_sub,CER020s,C. Visual arts (crafts) / F. Design,provisional
freq,NaN,63845,40813,316124,NaN,NaN,2197978,1171578,1171578,1487702,1333294
mean,1.098990e+06,NaN,NaN,NaN,2.013484e+03,1.537698e+01,NaN,NaN,NaN,NaN,NaN
std,6.345017e+05,NaN,NaN,NaN,6.494109e+00,3.154842e+02,NaN,NaN,NaN,NaN,NaN
min,1.000000e+00,NaN,NaN,NaN,2.002000e+03,0.000000e+00,NaN,NaN,NaN,NaN,NaN
25%,5.494952e+05,NaN,NaN,NaN,2.008000e+03,3.000000e-03,NaN,NaN,NaN,NaN,NaN
50%,1.098990e+06,NaN,NaN,NaN,2.014000e+03,3.700000e-02,NaN,NaN,NaN,NaN,NaN
75%,1.648484e+06,NaN,NaN,NaN,2.019000e+03,5.460000e-01,NaN,NaN,NaN,NaN,NaN


# **Construir a "ficha técnica" (entregável da aula):**

In [ ]:
ficha = {
    "arquivo": "trade_edges.csv",
    "linhas": len(edges),
    "colunas": list(edges.columns),
    "periodo": (edges["year"].min(), edges["year"].max()),
    "economias_distintas": edges["economy"].nunique(),
    "dominios": edges["fcs_domain"].unique().tolist(),
    "memoria_MB": round(edges.memory_usage(deep=True).sum() / 1e6, 1),
}
ficha

{'arquivo': 'trade_edges.csv',
 'linhas': 2197978,
 'colunas': ['edge_id',
  'economy',
  'partner',
  'product',
  'year',
  'value_usd_millions',
  'flow',
  'resolution',
  'cer_code',
  'fcs_domain',
  'mapping_status'],
 'periodo': (2002, 2024),
 'economias_distintas': 204,
 'dominios': ['C. Visual arts (crafts) / F. Design',
  'D. Books and press',
  'B. Performance and celebration / C. Visual arts',
  'E. Audiovisual and interactive media',
  'F. Design and creative services',
  'A. Cultural and natural heritage'],
 'memoria_MB': np.float64(1148.6)}

## 🧪 Exercícios Práticos — Aula 1

> **Como usar:** resolva cada exercício em uma célula de código logo abaixo do enunciado. Depois, leve o resultado para uma IA (Claude, ChatGPT, Gemini) usando o *prompt sugerido* — adapte-o com os seus próprios números. Cole a resposta da IA em uma célula de texto e escreva, em 2-3 linhas, se você concorda com ela e por quê.

---

### Exercício 1 — Radiografia do dataset
**🎯 Objetivo:** praticar `.shape`, `.dtypes`, `.info()`.

**📝 Tarefa:** Descubra quantas linhas, quantas colunas e qual o tipo de dado de cada coluna em `trade_edges.csv`. Identifique quais colunas são numéricas e quais são categóricas.

**🤖 Pergunte à IA:**
> "Estou analisando um dataset de comércio bilateral de bens criativos com estas colunas: [cole a lista de colunas e tipos]. Para cada coluna, me diga se ela deveria ser tratada como **fato** (numérica, somável) ou **dimensão** (categórica, usada para agrupar/filtrar), e por quê."

---

### Exercício 2 — O ano que mais exportou
**🎯 Objetivo:** praticar `groupby`, `sum` e ordenação.

**📝 Tarefa:** Agrupe `trade_edges` por `year` e some `value_usd_millions`. Qual foi o ano com maior valor total exportado? Existe alguma queda visível ao longo do tempo?

**🤖 Pergunte à IA:**
> "O total exportado em bens criativos por ano foi: [cole os números]. Que hipóteses econômicas explicariam picos ou quedas nesses anos específicos? Cite eventos históricos plausíveis entre 2002 e 2024."

---

### Exercício 3 — Quem lidera cada domínio
**🎯 Objetivo:** praticar filtro (boolean indexing ou `.query()`) e `groupby`.

**📝 Tarefa:** Escolha um dos sete domínios (`fcs_domain`). Filtre apenas esse domínio no último ano disponível (2024) e descubra as 5 economias (`economy`) com maior valor total exportado.

**🤖 Pergunte à IA:**
> "Estes são os 5 maiores exportadores de [domínio escolhido] em 2024: [cole a lista]. Do ponto de vista de estratégia de negócios, o que essa liderança sugere sobre vantagem competitiva desses países nesse setor específico?"

---

### Exercício 4 — O funil bate com o paper?
**🎯 Objetivo:** praticar contagem simples e autoconferência (self-check de proveniência).

**📝 Tarefa:** Confirme se `len(edges)` é igual a **2.197.978**, o número reportado no paper. Se não for, investigue o motivo (download incompleto? filtro aplicado sem querer?).

**🤖 Pergunte à IA:**
> "Por que é importante, em um projeto de dados profissional, conferir se a contagem de linhas de um dataset bate exatamente com o número documentado pelos autores originais? Que riscos existem se essa conferência não for feita?"

---

### Exercício 5 — Uma pergunta de negócio, uma hipótese
**🎯 Objetivo:** transformar curiosidade em pergunta testável — o primeiro passo de qualquer projeto de CD (*Business Understanding*).

**📝 Tarefa:** Em dupla, escrevam uma pergunta de negócio que gostariam de responder com esse acervo ao longo do semestre (ex.: *"o Brasil está ganhando ou perdendo espaço na exportação de artesanato?"*). Escrevam também uma hipótese inicial — um palpite, mesmo sem evidência ainda.

**🤖 Pergunte à IA:**
> "Minha pergunta de negócio é: [cole a pergunta]. Minha hipótese inicial é: [cole a hipótese]. Me ajude a reformular essa pergunta em termos de uma variável que eu poderia medir e comparar ao longo do tempo usando dados de comércio bilateral."

---

> 💡 **Dica geral:** a IA é uma consultora, não um oráculo. Sempre confira se a resposta dela é coerente com os números que você mesmo calculou.